# UKF sigma points: Cholesky against LDLᵀ

An unscented Kalman filter (UKF) doesn't linearize its model. It propagates
$2n + 1$ *sigma points* through it: the mean $\hat x$ and
$\hat x \pm \gamma s_j$, where the $s_j$ are the columns of a square root
$S$ of the state covariance, $P = S S^T$. Every step needs a new $S$.

There are two standard ways to get one:

- **Cholesky**, $P = L L^T$: take $S = L$.
- **LDLᵀ**, $P = L D L^T$, with $L$ unit lower triangular: take
  $S = L\,D^{1/2}$.

The C++ examples behind this notebook argue for LDLᵀ. Cholesky takes a square
root at every pivot, and they hold that those square roots lose precision once
$P$ becomes ill-conditioned, which can make the filter diverge. LDLᵀ takes none
during the factorization.

This notebook tests that claim. It factors matrices of known condition number,
then runs the C++ examples' filters with each factorization, in float64 and in
float32.

**Uses:**
- `mtl5.cholesky(P).L`, `mtl5.ldlt(P).L` and `.diagonal()`;
- `mtl5.generators.hilbert`;
- `mtl5.eigvalsh`.

**From C++:** [`ukf_cholesky_vs_ldlt/ukf_comparison.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/ukf_cholesky_vs_ldlt/ukf_comparison.cpp) and [`ukf_bearing_only/bearing_only_ukf.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/ukf_bearing_only/bearing_only_ukf.cpp).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import mtl5

# Chart styling: recessive axes and grid, ink for text, validated categorical hues.
SURFACE = "#fcfcfb"
INK, INK_2 = "#0b0b0b", "#52514e"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)

U64, U32 = 2.0**-53, 2.0**-24


def rel_residual(P, Q):
    return np.linalg.norm(P - Q) / np.linalg.norm(P)


def eigvals(P):
    return mtl5.eigvalsh(mtl5.matrix(np.array(P, dtype=np.float64)))


def cond(P):
    lam = np.abs(eigvals(P))
    return lam.max() / lam.min()

## The two square roots, at any condition number

The Hilbert matrix $H_n$ is symmetric positive definite, and its condition
number grows by about a factor of 30 with each step of $n$. At $n = 12$ it is
$10^{16}$: its smallest eigenvalue is about $u$ times its largest, the size of
the rounding errors in its entries.

For each $n$, both factorizations are formed and multiplied back out. A
residual of exactly 0 is drawn at $10^{-19}$.

In [ ]:
rows = []
for n in range(2, 14):
    H = mtl5.generators.hilbert(n).to_numpy()
    f = mtl5.ldlt(H)
    L1, D = np.asarray(f.L), f.diagonal()
    try:
        L = np.asarray(mtl5.cholesky(H).L)
        chol = rel_residual(H, L @ L.T)
    except RuntimeError:
        chol = None  # met a pivot that is not positive
    rows.append((n, cond(H), chol, rel_residual(H, L1 @ np.diag(D) @ L1.T), D.min()))

print(f"{'n':>3} {'cond(H)':>9} {'Cholesky':>10} {'LDLT':>10} {'min D':>10}")
for n, c, rc, rl, dmin in rows:
    print(f"{n:3d} {c:9.1e} {'raises' if rc is None else f'{rc:.1e}':>10} {rl:10.1e} {dmin:10.1e}")

ok = [r for r in rows if r[2] is not None]
fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.loglog(
    [r[1] for r in ok],
    [max(r[2], 1e-19) for r in ok],
    "o-",
    color=SERIES[0],
    label="Cholesky, ‖H − LLᵀ‖ / ‖H‖",
)
ax.loglog(
    [r[1] for r in rows],
    [max(r[3], 1e-19) for r in rows],
    "s--",
    color=SERIES[1],
    label="LDLᵀ, ‖H − LDLᵀ‖ / ‖H‖",
)
ax.axhline(U64, color=INK_2, lw=0.8, ls=":")
ax.text(1.5e1, U64 * 1.3, "u", color=INK_2, fontsize=9)
ax.set_ylim(1e-19, 1e-14)
ax.set_xlabel("cond(Hₙ)")
ax.set_ylabel("relative residual")
ax.set_title("Hilbert matrices, n = 2 … 13, float64", fontsize=10)
ax.legend(fontsize=8, loc="upper left")
ax.grid(True)
fig.tight_layout()
plt.show()

Measured, both residuals stay below the unit roundoff $u$, from condition
number 19 to $10^{16}$, and neither grows with it. The theory bounds them by a
modest multiple of $u$, independent of the condition number:

- **The scalar square roots are not the problem.** A square root is as well
  conditioned as an operation can be: a relative perturbation $\epsilon$ in its
  argument changes the result by $\epsilon / 2$. That is a statement about
  each pivot, not about the factor as a whole: for an ill-conditioned $P$, a
  tiny change to $P$ can change a column of its factor a lot. For
  $P = \mathrm{diag}(1, \epsilon)$, doubling $\epsilon$ changes $P$ by
  about $\epsilon$ but its second column by 41%.
- **Cholesky is backward stable** (Higham, *Accuracy and Stability of
  Numerical Algorithms*, Theorem 10.3). The computed $L$ is the exact factor of
  a matrix within $c\,n\,u\,\|P\|$ of $P$, for a small constant $c$,
  whatever $P$'s condition number. LDLᵀ
  without pivoting has the same kind of bound for a positive definite matrix.

So the residuals measure backward accuracy: each method's sigma points are the
exact ones for a covariance within rounding of $P$. Neither method gives more
accurate factor columns than the other, and neither can, since those columns
are ill-conditioned functions of $P$ itself.

$n = 13$ is the limit. $H_{13}$ is positive definite in exact arithmetic, but
its smallest eigenvalue, about $3 \times 10^{-18}$, is far below the rounding
errors of storing and factoring it, about $u\,\|H_{13}\| \approx 2 \times
10^{-16}$. To working precision it is neither definite nor indefinite: the last
pivot is rounding noise, and its sign depends on the order of operations and
on whether the compiler fuses multiply-adds.

- **Where it comes out negative** (Linux x86-64, where this page was built:
  $-6 \times 10^{-17}$), Cholesky raises and LDLᵀ's last $D$ entry is negative.
- **Where it comes out positive** (macOS arm64: $+2 \times 10^{-14}$), both
  factorizations complete, with residuals still below $u$.

Either way, neither residual grows: the failure is a sign decided by
rounding, not a loss of accuracy.

## Where they differ: an indefinite P

That is the one real difference. A covariance update
$P \leftarrow P^- - K S K^T$ can round to a matrix with a slightly negative
eigenvalue. Below, $H_4$ is shifted by its smallest eigenvalue plus $\delta$,
so the result has an eigenvalue of $-\delta$.

In [ ]:
H4 = mtl5.generators.hilbert(4).to_numpy()
lam_min = eigvals(H4).min()
inertia = []
print(f"{'delta':>7} {'Cholesky':>10} {'D from LDLT':>48} {'negative eigenvalues':>21}")
for delta in (1e-14, 1e-10, 1e-6, 1e-2):
    P = H4 - (lam_min + delta) * np.eye(4)
    try:
        mtl5.cholesky(P)
        chol = "factors"
    except RuntimeError:
        chol = "raises"
    D = mtl5.ldlt(P).diagonal()
    neg_eig = int((eigvals(P) < 0).sum())
    inertia.append((chol, int((D < 0).sum()), neg_eig))
    print(f"{delta:7.0e} {chol:>10} {np.array2string(D, precision=2):>48} {neg_eig:21d}")

Cholesky raises, as it must: no real $L$ satisfies $L L^T = P$. Here every
pivot is nonzero, so LDLᵀ returns a factorization, and by Sylvester's law of
inertia, $D$ has as many negative entries as $P$ has negative eigenvalues.
That needs the factorization to exist. LDLᵀ does not pivot, so an indefinite
matrix with a zero pivot, such as $\begin{pmatrix} 0 & 1 \\ 1 & 0
\end{pmatrix}$, makes it raise as well; `mtl5.bunch_kaufman` handles those. At $\delta = 10^{-2}$ the shift passes
$H_4$'s second eigenvalue as well, and two entries are negative.

So LDLᵀ *detects* indefiniteness and leaves the response to the filter. It
doesn't give sigma points for $P$, because $D^{1/2}$ doesn't exist. The C++
range-and-bearing example uses $|D|$, which gives sigma points for a different,
positive definite matrix. The bearing-only example stops the filter instead.

## The filter

This is the UKF of both C++ examples. Weighted sums are accumulated
sequentially in the filter's precision, as in the C++. The square root is the
only thing that changes between the two methods. For LDLᵀ, a $D$ entry that
isn't positive counts as a failure, as in the bearing-only example.

One change from the C++: both trackers start with the target near the beacon's
$-x$ axis, where `atan2` jumps from $\pi$ to $-\pi$. Sigma points on either side
of that cut have nearly equal bearings that differ by almost $2\pi$ as numbers.
So before the mean and the covariances are formed, a sigma point's bearing
more than $\pi$ from the central point's is moved by $2\pi$. The other
bearings keep their values bit for bit. This changes the measurement transform:
when the points straddle the cut, it changes the result even in exact
arithmetic, and wrapping the final innovation, as the C++ does, can't undo that.

Three smaller differences round differently, which matters in float32:

- **The sums:** the notebook adds $Q$ and $R$ after summing the weighted outer
  products (the C++ starts its sums from them), and it forms each outer product
  before weighting it (the C++ weights first).
- **The gain:** the notebook solves $K S = P_{xz}$ with `np.linalg.solve`. The
  bearing-only C++ divides by its scalar $S$, and the range-and-bearing C++
  inverts the 2×2 $S$ explicitly and stops if its determinant is below
  $10^{-30}$. That check never fires here.

In [ ]:
class Diverged(Exception):
    pass


def weights(n, alpha, dtype, beta=2.0, kappa=0.0):
    T = np.dtype(dtype).type
    lam = T(alpha) * T(alpha) * (T(n) + T(kappa)) - T(n)
    Wm = np.full(2 * n + 1, T(1) / (T(2) * (T(n) + lam)), dtype=dtype)
    Wc = Wm.copy()
    Wm[0] = lam / (T(n) + lam)
    Wc[0] = Wm[0] + (T(1) - T(alpha) * T(alpha) + T(beta))
    return np.sqrt(T(n) + lam), Wm, Wc


def square_root(P, method):
    # S with P = S S^T, or Diverged.
    try:
        if method == "cholesky":
            return np.asarray(mtl5.cholesky(P).L)
        f = mtl5.ldlt(P)
    except RuntimeError as e:
        raise Diverged(method) from e
    D = f.diagonal()
    if (D <= 0).any():
        raise Diverged(method)
    return np.asarray(f.L) * np.sqrt(D).astype(P.dtype)


def sigma_points(x, S, gamma):
    return np.vstack([x, x + gamma * S.T, x - gamma * S.T])


def wsum(W, X):
    acc = W[0] * X[0]
    for w, row in zip(W[1:], X[1:]):
        acc = acc + w * row
    return acc


def wcov(W, A, B):
    acc = W[0] * np.outer(A[0], B[0])
    for w, a, b in zip(W[1:], A[1:], B[1:]):
        acc = acc + w * np.outer(a, b)
    return acc


def mean_bias(x, P, method, gamma, Wm):
    # How far the weighted mean of the sigma points lands from x, relative to
    # |x|. The points are symmetric about x, so in exact arithmetic it is 0.
    X = sigma_points(x, square_root(P, method), gamma)
    return float(np.linalg.norm(wsum(Wm, X) - x) / np.linalg.norm(x))


def run_ukf(model, method, dtype=np.float64, alpha=1e-3, steps=None):
    T = np.dtype(dtype).type
    steps = steps or model["steps"]
    gamma, Wm, Wc = weights(4, alpha, dtype)
    f, h, Q = model["f"], model["h"], model["Q"].astype(dtype)
    x, P = model["x0"].astype(dtype), model["P0"].astype(dtype)
    truth, zs = model["truth"], model["z"]
    out = {"err": [], "cond": [], "bias": [], "x": [], "failed_at": None, "failed_on": None}
    for k in range(steps):
        R = model["R"](k).astype(dtype)
        stage = "incoming P"  # the previous step's update
        try:
            out["bias"].append(mean_bias(x, P, method, gamma, Wm))
            X = np.array(
                [f(s) for s in sigma_points(x, square_root(P, method), gamma)], dtype=dtype
            )
            xp = wsum(Wm, X)
            Pp = Q + wcov(Wc, X - xp, X - xp)
            stage = "predicted P"
            X = sigma_points(xp, square_root(Pp, method), gamma)
        except Diverged:
            out["failed_at"], out["failed_on"] = k, stage
            break
        Z = np.array([h(s) for s in X], dtype=dtype)
        # The bearing is an angle: take each sigma point's bearing as an offset
        # from the central one, wrapped, so points that straddle the atan2
        # branch cut at +-pi average to the right mean and spread.
        # Only a bearing more than pi from the central one is moved, by 2 pi;
        # the rest keep their values bit for bit.
        off = Z[:, -1] - Z[0, -1]
        Z[off > T(np.pi), -1] -= T(2 * np.pi)
        Z[off < T(-np.pi), -1] += T(2 * np.pi)
        zp = wsum(Wm, Z)
        Szz = R + wcov(Wc, Z - zp, Z - zp)
        Pxz = wcov(Wc, X - xp, Z - zp)
        K = np.linalg.solve(Szz.T, Pxz.T).T
        innov = zs[k].astype(dtype) - zp
        innov[-1] = T((float(innov[-1]) + np.pi) % (2 * np.pi) - np.pi)  # the bearing wraps
        x = xp + K @ innov
        P = Pp - K @ Szz @ K.T
        P = np.triu(P) + np.triu(P, 1).T  # symmetric, as in the C++
        out["err"].append(float(np.linalg.norm(truth[k] - x)))
        out["cond"].append(cond(P))
        out["x"].append(x.astype(np.float64))
    return out

## Range and bearing (`ukf_comparison`)

The first example tracks a target moving with quadratic drag, from range and
bearing measurements taken at a beacon at (10, 0). Its knob is the measurement
noise $R$. The example expects small $R$ to drive $P$ to ill-conditioning, and
Cholesky to degrade or diverge there. Its four scenarios run 200 steps each. The
true trajectory and the noise are drawn once, with a fixed seed, and both
filters see the same measurements.

In [ ]:
DT, DRAG = 0.1, 0.01


# The models convert their constants to the state's type first, so a float32
# filter computes in float32 under NumPy 1.x as well, where a NumPy scalar
# times a Python float is promoted to float64.
def drag_model(x):
    T = x.dtype.type
    dt = T(DT)
    damp = max(T(1) - T(DRAG) * np.hypot(x[2], x[3]) * dt, T(0.1))
    return np.array([x[0] + x[2] * dt, x[1] + x[3] * dt, x[2] * damp, x[3] * damp], dtype=x.dtype)


def range_bearing(x, beacon=(10.0, 0.0)):
    T = x.dtype.type
    dx, dy = x[0] - T(beacon[0]), x[1] - T(beacon[1])
    return np.array([np.hypot(dx, dy), np.arctan2(dy, dx)], dtype=x.dtype)


def simulate(f, h, x_true, Q, R, steps, seed):
    rng = np.random.default_rng(seed)
    truth, z = [], []
    for k in range(steps):
        x_true = f(x_true) + np.sqrt(np.diag(Q)) * rng.standard_normal(4)
        truth.append(x_true)
        z.append(h(x_true) + np.sqrt(np.diag(R(k))) * rng.standard_normal(len(np.diag(R(k)))))
    return np.array(truth), np.array(z)


def comparison_model(R, steps=200, seed=12345):
    Q = np.diag([1e-4, 1e-4, 1e-2, 1e-2])
    truth, z = simulate(drag_model, range_bearing, np.array([0, 0, 1, 0.5]), Q, R, steps, seed)
    return dict(
        f=drag_model,
        h=range_bearing,
        Q=Q,
        R=R,
        steps=steps,
        truth=truth,
        z=z,
        x0=np.array([0.5, -0.5, 0.8, 0.3]),
        P0=np.eye(4),
    )


scenarios = {
    "benign": lambda k: np.diag([1.0, 1.0]),
    "moderate": lambda k: np.diag([1e-2, 1e-1]),
    "severe": lambda k: np.diag([1e-14, 1e-2]),
    "intermittent": lambda k: np.diag([1e-8, 1e-2]) if (k // 25) % 2 == 0 else np.diag([1.0, 1.0]),
}


def gap(runs):
    # The largest difference between the two estimates, or NaN if either
    # filter stopped early.
    if any(r["failed_at"] is not None for r in runs.values()):
        return float("nan")
    return np.abs(np.array(runs["cholesky"]["x"]) - np.array(runs["ldlt"]["x"])).max()


comparison = {}
print(f"{'scenario':>12} {'max cond(P)':>12} {'final error':>12}", end="")
print(f" {'|Cholesky - LDLT|':>18} {'same, alpha = 1':>16}")
for name, R in scenarios.items():
    model = comparison_model(R)
    runs = {m: run_ukf(model, m) for m in ("cholesky", "ldlt")}
    runs_1 = {m: run_ukf(model, m, alpha=1.0) for m in ("cholesky", "ldlt")}
    comparison[name] = (runs, runs_1, gap(runs), gap(runs_1))
    stopped = {
        f"{m}, alpha = {a}": r["failed_at"]
        for a, rs in ((1e-3, runs), (1, runs_1))
        for m, r in rs.items()
        if r["failed_at"] is not None
    }
    if stopped:
        print(f"{name:>12} stopped early: {stopped}")
        continue
    c = runs["ldlt"]
    print(f"{name:>12} {max(c['cond']):12.1e} {c['err'][-1]:12.3f}", end="")
    print(f" {gap(runs):18.1e} {gap(runs_1):16.1e}")

In every scenario, both filters run all 200 steps, and their estimates agree to
about $10^{-7}$. That gap is rounding error, amplified by the UKF's weights
(see the weights section below). With $\alpha = 1$ instead of the examples'
$10^{-3}$, the two filters agree to about $10^{-13}$. The covariance never
gets ill-conditioned enough to trouble either factorization: its condition
number stays below $10^4$, even at $R = 10^{-14}$.

## Bearing only, in float64 and float32 (`bearing_only_ukf`)

The second example is the classic hard case. Bearing-only measurements, with
$R = 10^{-4}$, observe the direction to the target well and its range not at
all, so $P$ is stretched along the line of sight. It runs 15 steps in float64
and in float32.

In [ ]:
def constant_velocity(x):
    dt = x.dtype.type(DT)
    return np.array([x[0] + x[2] * dt, x[1] + x[3] * dt, x[2], x[3]], dtype=x.dtype)


def bearing(x, beacon=(50.0, 0.0)):
    T = x.dtype.type
    return np.array([np.arctan2(x[1] - T(beacon[1]), x[0] - T(beacon[0]))], dtype=x.dtype)


Q_b = np.diag([1e-6, 1e-6, 1e-4, 1e-4])
R_b = lambda k: np.array([[1e-4]])  # noqa: E731
truth_b, z_b = simulate(constant_velocity, bearing, np.array([0, 5, 2, -0.5]), Q_b, R_b, 15, 54321)
bearing_model = dict(
    f=constant_velocity,
    h=bearing,
    Q=Q_b,
    R=R_b,
    steps=15,
    truth=truth_b,
    z=z_b,
    x0=np.array([1, 6, 1.5, 0.0]),
    P0=100 * np.eye(4),
)

bearing_runs = {
    (np.dtype(dt).name, m): run_ukf(bearing_model, m, dt)
    for dt in (np.float64, np.float32)
    for m in ("cholesky", "ldlt")
}

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for (dt, m), r in bearing_runs.items():
    color = SERIES[0] if dt == "float64" else SERIES[1]
    style = "-" if m == "cholesky" else "--"
    axes[0].semilogy(
        range(len(r["bias"])), r["bias"], style, color=color, marker="o", ms=3, label=f"{dt}, {m}"
    )
    axes[1].semilogy(range(len(r["cond"])), r["cond"], style, color=color, marker="o", ms=3)
axes[0].set_title("weighted mean of the sigma points: |mean − x̂| / |x̂|", fontsize=10)
axes[1].set_title("cond(P) after each update", fontsize=10)
for ax in axes:
    ax.set_xlabel("step")
    ax.grid(True)
axes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()
for key, r in bearing_runs.items():
    print(
        f"{key[0]:>8} {key[1]:>9}: "
        + (
            f"fails at step {r['failed_at']}, factoring the {r['failed_on']}"
            if r["failed_at"] is not None
            else f"runs {len(r['err'])} steps, final error {r['err'][-1]:.3f}"
        )
    )

Two things stand out.

- **In float64, Cholesky and LDLᵀ behave the same.** Each line has a twin, and
  both filters end with the same error.
- **In float32, both filters fail at the same step, on the same matrix.**
  On the page's build it is the incoming covariance, the previous step's
  update $P^- - K S K^T$: it is no longer positive definite, so Cholesky raises and LDLᵀ shows a $D$ entry that
  isn't positive. The C++ status columns test that same incoming $P$.
- **The float32 bias is large from step 0**, when $P = 100 I$, a perfectly
  conditioned matrix. The weighted mean of the sigma points lands a few percent
  of $|\hat x|$ away from $\hat x$. Neither factorization can be the cause:
  any square root gives points symmetric about $\hat x$.

## What breaks float32: the weights

The examples use the common scaling $\alpha = 10^{-3}$. With $n = 4$ states,
$\lambda = \alpha^2 n - n \approx -4$, and the weights are

$$W_0 = \frac{\lambda}{n + \lambda} \approx -10^6, \qquad
  W_i = \frac{1}{2(n + \lambda)} \approx 1.25 \times 10^5 .$$

They sum to 1, but each term of the weighted mean is about $10^6$ times larger
than the mean. The sum cancels away 6 of float32's 7 digits, and in float32
even $\lambda$ itself has lost digits. The covariance weights cancel the same
way, in $P^-$, $S$ and $P_{xz}$. The update $P^- - K S K^T$ then subtracts
two nearly equal matrices built from them, and its result loses positive
definiteness. The C++ program's float32 covariance fails at step 5 instead of
step 3: its noise and the order of its sums and its gain differ, and at these
weights the rounding decides when.

In [ ]:
x0_32, P0_32 = bearing_model["x0"].astype(np.float32), bearing_model["P0"].astype(np.float32)
alphas = (1e-3, 1e-2, 1e-1, 1.0)
print(f"{'alpha':>6} {'W0 (float32)':>14} {'step-0 bias, Cholesky':>22} {'LDLT':>9}")
bias0 = {}
for a in alphas:
    gamma, Wm, _ = weights(4, a, np.float32)
    bias0[a] = [mean_bias(x0_32, P0_32, m, gamma, Wm) for m in ("cholesky", "ldlt")]
    print(f"{a:6.0e} {Wm[0]:14.6g} {bias0[a][0]:22.1e} {bias0[a][1]:9.1e}")

fixed = {m: run_ukf(bearing_model, m, np.float32, alpha=1.0) for m in ("cholesky", "ldlt")}
ref64 = run_ukf(bearing_model, "cholesky", np.float64, alpha=1.0)
for m, r in fixed.items():
    print(
        f"float32, alpha = 1, {m:>8}: "
        + (
            f"fails at step {r['failed_at']}"
            if r["failed_at"] is not None
            else f"runs {len(r['err'])} steps, final error {r['err'][-1]:.3f}"
        )
    )
print(f"float64, alpha = 1, cholesky: final error {ref64['err'][-1]:.3f}")

With $\alpha = 1$, the weights are of order 1 ($W_0 = 0$), the bias
vanishes, and the float32 filter runs to the end with either factorization. Its
estimate is as good as the float64 filter's with the same $\alpha$. (Other remedies keep the small $\alpha$, such as forming the
mean about $\sigma_0$ or accumulating the sums in higher precision.) The
precision the filter lost was in the weights, not in a square root.

## Check

- **Backward stability:** for $H_2$ to $H_{12}$, condition numbers up to
  $10^{16}$, both residuals are below $u$.
- **The limit:** $H_{13}$'s smallest $D$ entry is below $10^{-12}$ in size.
  Cholesky raises exactly when that entry isn't positive, and any residual it
  does compute is below $u$.
- **Inertia:** on the shifted matrices, Cholesky raises, and the number of
  negative $D$ entries equals the number of negative eigenvalues.
- **Range and bearing:** in all four scenarios, all four filters (each
  factorization, with $\alpha = 10^{-3}$ and with $\alpha = 1$) run all 200
  steps. The two factorizations' estimates differ by less than $10^{-6}$, and
  with $\alpha = 1$ by less than $10^{-12}$.
- **Bearing only:**
  - In float64, both filters complete, and their final errors agree to
    $10^{-6}$.
  - In float32, the two filters fail at the same step on the same matrix, or
    both complete.
  - With $\alpha = 10^{-3}$, the float32 step-0 bias is above $10^{-3}$, and
    the same for both factorizations.
  - With $\alpha = 1$, it is below $10^{-6}$, both float32 filters complete,
    and their final errors are within 1% of the float64 filter's.

In [ ]:
for n, c, rc, rl, dmin in rows:
    if n <= 12:
        assert rc is not None and rc < U64 and rl < U64, n
n, c, rc, rl, dmin = rows[-1]
assert n == 13 and abs(dmin) < 1e-12 and rl < U64
assert (rc is None) == (dmin <= 0) and (rc is None or rc < U64)

for chol, neg_d, neg_eig in inertia:
    assert chol == "raises" and neg_d == neg_eig >= 1

for name, (runs, runs_1, g, g_1) in comparison.items():
    for r in list(runs.values()) + list(runs_1.values()):
        assert r["failed_at"] is None and len(r["x"]) == 200, name
    assert g < 1e-6 and g_1 < 1e-12, (name, g, g_1)

assert all(bearing_runs[("float64", m)]["failed_at"] is None for m in ("cholesky", "ldlt"))
r32c, r32l = bearing_runs[("float32", "cholesky")], bearing_runs[("float32", "ldlt")]
assert (r32c["failed_at"], r32c["failed_on"]) == (r32l["failed_at"], r32l["failed_on"])
e64 = [bearing_runs[("float64", m)]["err"][-1] for m in ("cholesky", "ldlt")]
assert abs(e64[0] - e64[1]) < 1e-6 * e64[0]
assert min(bias0[1e-3]) > 1e-3 and np.isclose(*bias0[1e-3], rtol=0.1)
assert max(bias0[1.0]) < 1e-6
assert all(r["failed_at"] is None for r in fixed.values())
assert all(abs(r["err"][-1] - ref64["err"][-1]) < 1e-2 * ref64["err"][-1] for r in fixed.values())
print("all checks passed")

## From C++

[`ukf_cholesky_vs_ldlt/ukf_comparison.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/ukf_cholesky_vs_ldlt/ukf_comparison.cpp) and [`ukf_bearing_only/bearing_only_ukf.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/ukf_bearing_only/bearing_only_ukf.cpp) run these filters, with the same
models, noise levels and initial states. Differences:

- **Noise:** the C++ programs draw it from their own generator, so the
  trajectories differ in detail.
- **Bearing only:** the C++ advances one filter with LDLᵀ and, at each step,
  tries both factorizations on its $P$. Here each method runs its own filter.
- **The bearing's branch cut:** handled in the sigma points here, as described
  under "The filter"; the C++ wraps only the innovation.
- **Their conclusions don't survive their own output.** Run, the C++ programs
  report:
  - **Range and bearing:** in the scenarios with a per-step table (benign,
    moderate, severe), Cholesky and LDLᵀ give the same estimates. Neither
    diverges, at $R = 10^-14$ either, and the intermittent scenario completes
    with both.
  - **Bearing only:** in float32, both factorizations fail on the same
    covariance from step 5 on. (That program advances a single filter with
    LDLᵀ, so this is one covariance tested twice, not two filters.) The
    "sigma-point bias" is the same for both.

  The comparisons of two independent filters are this notebook's.